In [ ]:
import os 
from dotenv import load_dotenv 
load_dotenv()

True

In [2]:
if os.environ["OPENAI_API_KEY"]:
    print("API key is set")

API key is set


In [3]:
from langchain_openai import ChatOpenAI

In [5]:
llm= ChatOpenAI(model='gpt-5-nano',temperature=0) # to send requuest to model , temp=0 no creative answers

**RAG IMPLEMENTATION WITH OWNN PDF**

**STEP 1 Extracting text from PDF

In [6]:
from langchain_community.document_loaders import PyPDFLoader

pdf_path= "./fabric-admin.pdf"

loader=PyPDFLoader(pdf_path)
docs=loader.load()

docs

/var/folders/km/n5txgrn17132qnzvdz2swxbc0000gn/T/ipykernel_1638/3132491757.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


[Document(metadata={'producer': 'Microsoft Learn PDF 1.0.26144.02', 'creator': 'Microsoft Learn', 'creationdate': '2026-07-13T16:01:57+00:00', 'title': 'fabric admin | Microsoft Learn', 'moddate': '2026-07-13T16:01:57+00:00', 'source': './fabric-admin.pdf', 'total_pages': 316, 'page': 0, 'page_label': '1'}, page_content='Fabric administration documentation\nLearn about the Fabric admin settings, options, and tools.\nFabric in your organization\nｅOVERVIEW\nWhat is administration in Fabric?\nｂGET STARTED\nEnable Fabric for your organization\nRegion availability\nFind your Fabric home region\nｃHOW-TO GUIDE\nUnderstand Fabric admin roles\nｉREFERENCE\nGovernance documentation\nSecurity documentation\nTools and settings\nｅOVERVIEW\nAbout tenant settings\nｃHOW-TO GUIDE\nSet up git integration\nSet up item certification\nConfigure notifications\nSet up metadata scanning'),
 Document(metadata={'producer': 'Microsoft Learn PDF 1.0.26144.02', 'creator': 'Microsoft Learn', 'creationdate': '2026-07

**Step 1.1 Creating own meta data for PDF CHUNKS

In [7]:
for i in docs:
    i.metadata= {"source":"fabric-admin.pdf",
                 "developer":'microsoft'}

In [7]:
docs

[Document(metadata={'source': 'fabric-admin.pdf', 'developer': 'microsoft'}, page_content='Fabric administration documentation\nLearn about the Fabric admin settings, options, and tools.\nFabric in your organization\nｅOVERVIEW\nWhat is administration in Fabric?\nｂGET STARTED\nEnable Fabric for your organization\nRegion availability\nFind your Fabric home region\nｃHOW-TO GUIDE\nUnderstand Fabric admin roles\nｉREFERENCE\nGovernance documentation\nSecurity documentation\nTools and settings\nｅOVERVIEW\nAbout tenant settings\nｃHOW-TO GUIDE\nSet up git integration\nSet up item certification\nConfigure notifications\nSet up metadata scanning'),
 Document(metadata={'source': 'fabric-admin.pdf', 'developer': 'microsoft'}, page_content='Enable content certification\nEnable service principal authentication\nConfigure Multi-Geo support\nMonitoring and management\nｅOVERVIEW\nWhat is the admin monitoring workspace?\nｐCONCEPT\nFeature usage and adoption report\nｃHOW-TO GUIDE\nUse the Monitoring hub\n

** Step 2 Chunking
splitting documents to chunks

In [8]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document  

splitter= RecursiveCharacterTextSplitter(
    chunk_size=1000, 
    chunk_overlap=100
)

chunks= splitter.split_documents(docs)
chunks

[Document(metadata={'source': 'fabric-admin.pdf', 'developer': 'microsoft'}, page_content='Fabric administration documentation\nLearn about the Fabric admin settings, options, and tools.\nFabric in your organization\nｅOVERVIEW\nWhat is administration in Fabric?\nｂGET STARTED\nEnable Fabric for your organization\nRegion availability\nFind your Fabric home region\nｃHOW-TO GUIDE\nUnderstand Fabric admin roles\nｉREFERENCE\nGovernance documentation\nSecurity documentation\nTools and settings\nｅOVERVIEW\nAbout tenant settings\nｃHOW-TO GUIDE\nSet up git integration\nSet up item certification\nConfigure notifications\nSet up metadata scanning'),
 Document(metadata={'source': 'fabric-admin.pdf', 'developer': 'microsoft'}, page_content='Enable content certification\nEnable service principal authentication\nConfigure Multi-Geo support\nMonitoring and management\nｅOVERVIEW\nWhat is the admin monitoring workspace?\nｐCONCEPT\nFeature usage and adoption report\nｃHOW-TO GUIDE\nUse the Monitoring hub\n

In [9]:
chunks[0].metadata

{'source': 'fabric-admin.pdf', 'developer': 'microsoft'}

In [10]:
len(chunks)

574

** Step3 Creating embedding for the chunks

In [11]:
from langchain_openai import OpenAIEmbeddings

embedding_model= OpenAIEmbeddings(model='text-embedding-3-small') # by default model



** Step 4 Storage
Storing embedding in the existing local vecor store
1) We will add documents to existing vecorstore

CREATE AND STORE EMBEDDING IN LOCAL VECTOR STORE

In [12]:

               
from langchain_community.vectorstores import Chroma

vectorstore = Chroma(
    persist_directory='./Vector/',
    embedding_function=embedding_model
)


/var/folders/km/n5txgrn17132qnzvdz2swxbc0000gn/T/ipykernel_1638/1886050944.py:3: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectorstore = Chroma(


** Step 4.1 Now we will add more documents to existing vector store

In [13]:
vectorstore.add_documents(chunks)

['4bcf3cce-1267-4596-b0e1-d9ac886595ff',
 'bc103f03-b251-4773-9003-03952e089911',
 'd013ce84-3ed9-4291-8556-aaebb95bfb06',
 'ae1b3b64-6dee-4edc-b676-912147c64eba',
 '22342e3c-a445-4b9f-b62b-9ea5197da514',
 '7a8050e8-35ec-4534-aa39-ea32a15a073a',
 '01703613-706a-47b0-ae00-3cedf7c81da7',
 '17150831-92bc-43a1-a38b-f2e21d55db46',
 '4ffaa165-c7e1-44dd-a6c8-ed686003758a',
 '8daad5a9-08dc-4300-b961-d6a1f648fd94',
 '05e7bd32-2f5f-4ddd-91ed-5e7962856b57',
 'be4c19aa-96f1-4dad-aaed-841e474bc924',
 'b2bfe7dc-6954-4277-befb-3265b97710bf',
 '21abee19-72e7-4ade-bea9-082d69ad0b1a',
 '25957fd0-20a8-46ca-b01c-faf0793d5f5d',
 'eb0211ff-ccb6-4201-96d8-c1c127acd7ba',
 '2dc7fed0-e65c-4d4a-a8d0-833f0e9793e7',
 '703a5681-8648-4b40-bad8-5c0926fe671f',
 'e3071327-8708-4917-80da-777879b0c0f8',
 '00a7ab98-fbce-4097-8fa1-3ee57d6e9072',
 '167a40e4-1eef-4f2f-a253-41ceae3fd781',
 '30a83980-c974-400e-9f7f-61d1424b3df8',
 '5f33dc77-1915-411a-bf2a-c57dd70e9648',
 '95870b07-1742-4e89-9ce8-556ee0dc943f',
 '71311222-967c-

We have chunks provided by user+ embedding model (open AI model)+ metadata ==

Both these inputs are provided to langchain

Langchain will convert these inputs to 5 vector above 

and these 5 vetcor will save to Chroma DB automatically ]

it has run below loop tp save these vectors one by one along with metadeta

vector=[]
for doc in chunks :
vector=embedding_model.embed_documents([doc.page_content])
vector.append(vector)


Chroma DB will store vector+metadata   (Metadata+text)
like below
Document(Metadata={},page_content="_")

meaning storing vector+mapped_chunk


** Step5 Similarity search/ Semantic search

In [14]:
vectorstore.similarity_search('What is micrsoft fabric admin',k=3)

[Document(metadata={'developer': 'microsoft', 'source': 'fabric-admin.pdf'}, page_content='What is Microsoft Fabric administration?\nFabric administration is the set of tasks and tools you use to configure, secure, and govern the\nFabric software as a service (SaaS) platform across your organization. As an admin, you control\ntenant-wide settings, manage feature access to meet company policies and regulations, and\ndelegate responsibilities so no single team becomes a bottleneck.\nThere are generally three categories of tasks that admins focus on to ensure the platform is\nconfigured correctly and compliant with organizational policies:\nAdministration—This Fabric administration documentation covers how to manage the\nFabric platform, configure tenant and workspace features, and monitor usage and activity.\nSecurity—See the Security documentation to learn how to help safeguard data with identity,\naccess, encryption, and network protection settings.\nGovernance—See the Governance docum

I have Chroma DB . I am user and do similarity search as above code

Chorma DB will return 3 vector 

my question will first converted to verctor store using model an this vector will do the similary serach in Chorma DB and then it will return top 3 vectors and these top 3 vectors will return mapped chunks and then it will drive the result 

** Step6 Talk to LLM

In [15]:
context=vectorstore.similarity_search('Why do we need one lake?',k=3)

In [16]:
llm.invoke(f'Why do we need one lake?, you can answer using folliwing context: {context}')

AIMessage(content='Here’s why having OneLake is useful, based on the provided Fabric docs:\n\n- Central, cross-tool data access\n  - OneLake serves as a single data lake that Fabric apps (Spark, Data Engineering, Data Warehouse) and external apps (ADLS APIs, OneLake file explorer, Databricks) can access. This helps avoid data silos and simplifies how teams work with the same data.\n\n- Secure, time-bound access\n  - Access to OneLake is controlled with short-lived user-delegated SAS tokens, limited by least privilege and a maximum lifetime of one hour. This provides strong, auditable security for data sharing and access.\n\n- Easy data movement and reuse\n  - Semantic models can export data to OneLake, and once there, those tables can be included in Fabric items like lakehouses and warehouses. This makes it easy to reuse and share data across different Fabric components.\n\n- Admin control and governance\n  - OneLake settings are managed at the tenant level, including who can access On

In [17]:
response=llm.invoke(f'Why do we need one lake?,, you can answer using folliwing context: {context}')
response.content

'You need OneLake because it gives you a centralized, secure, and interoperable data store that works across Fabric and outside apps. Based on the provided document excerpts, here’s why it’s useful:\n\n- Unified data access for both Fabric apps and external tools\n  - You can access OneLake data from apps inside Fabric (Spark, Data Engineering, Data Warehouse) as well as outside Fabric via ADLS APIs, OneLake file explorer, and Databricks. This reduces data silos and makes data more reusable.\n\n- Secure, time-limited access\n  - Access can be granted through OneLake SAS tokens that are short-lived (up to one hour) and constrained by least-privilege permissions tied to a Fabric user’s Entra identity. This helps control who can read data and for how long.\n\n- Data export and reuse in Fabric\n  - Semantic models configured for OneLake can export/import tables to OneLake. Once in OneLake, those data assets can be included in Fabric items like lakehouses and warehouses, enabling consistent

** Step 7 Resuse the vector DB

In [21]:
embedding = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

vectorstore_persist = Chroma(
    persist_directory="./Vector",
    embedding_function=embedding
)

In [22]:
vectorstore_persist.similarity_search('Why do we need one lake, k=3)')

[Document(metadata={'developer': 'microsoft', 'source': 'fabric-admin.pdf'}, page_content="(ADLS) APIs, OneLake File Explorer, and Databricks. Users can already access data\nstored in OneLake with apps internal to the Fabric environment, such as Spark,\nData Engineering, and Data Warehouse. Learn More\nUse short-lived user-\ndelegated SAS tokens\nOneLake SAS tokens enable applications to access data in OneLake through short-\nlived SAS tokens, based on a Microsoft Fabric user's Entra identity. These token's\npermissions can be further limited to provide least privileged access and cannot\nexceed a lifetime of one hour. Learn More\nDatamart settings\nﾉ Expand table\nSemantic model settings\nﾉ Expand table\nScale-out settings\nﾉ Expand table\nOneLake settings\nﾉ Expand table"),
 Document(metadata={'developer': 'microsoft', 'source': 'fabric-admin.pdf'}, page_content='export data to OneLake\nSemantic models configured for OneLake integration can send import tables to\nOneLake. Once the da